# импорты всех необходимых мне библиотек

In [ ]:
import os, gc, re, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer

# Параметры + config путей к файлам

In [ ]:
RANDOM_SEED = 42
N_FOLDS = 5

TRAIN_PATH = "../input/datasets-avito-test/train-dset.parquet"
TEST_PATH  = "../input/datasets-avito-test/test-dset-small.parquet"

USE_COLS_TRAIN = [
    "query_id", "item_id",
    "query_text", "item_title", "item_description",
    "query_cat", "query_mcat", "query_loc",
    "item_cat_id", "item_mcat_id", "item_loc",
    "price", "item_query_click_conv",
    "item_contact"
]
USE_COLS_TEST = [c for c in USE_COLS_TRAIN if c != "item_contact"]


In [ ]:
print("Loading data...")
train = pd.read_parquet(TRAIN_PATH, columns=USE_COLS_TRAIN)
test  = pd.read_parquet(TEST_PATH,  columns=USE_COLS_TEST)
print("Data loaded!")

# не сильно помогло на практике, но оптимизируем память

In [ ]:
def optimize_memory(df):
    for c in df.columns:
        if c in ("query_text","item_title","item_description"):
            continue
        if df[c].dtype == "object":
            continue
        if pd.api.types.is_integer_dtype(df[c]):
            df[c] = df[c].astype(np.int32)
        elif pd.api.types.is_float_dtype(df[c]):
            df[c] = df[c].astype(np.float32)
    return df

train = optimize_memory(train)
test  = optimize_memory(test)


# Чистим весь текст от мусора

In [ ]:
_bad = re.compile(r"[^a-zа-я0-9 ]", re.I)
_sp  = re.compile(r"\s+")

def clean_text_chunked(series, batch_size=200_000):
    out = []
    for i in range(0, len(series), batch_size):
        s = series.iloc[i:i+batch_size].fillna("").str.lower()
        s = s.str.replace(_bad, " ", regex=True)
        s = s.str.replace(_sp, " ", regex=True).str.strip()
        out.append(s)
    return pd.concat(out)

for c in ["query_text","item_title","item_description"]:
    train[c] = clean_text_chunked(train[c])
    test[c]  = clean_text_chunked(test[c])
    gc.collect()

# Создаю дополнительные фичи, какие-то будут использоваться в будущем, какие-то просто для получения из них доп фичей

In [ ]:
def build_basic(df, price_median=None):
    price = df["price"].astype(np.float32)
    if price_median is None:
        price_median = float(np.nanmedian(price))
    price = price.fillna(price_median)

    click = df["item_query_click_conv"].astype(np.float32).fillna(0).clip(0,1)

    qlen = df["query_text"].str.len().astype(np.float32)
    tlen = df["item_title"].str.len().astype(np.float32)
    dlen = df["item_description"].str.len().astype(np.float32)

    feats = pd.DataFrame({
        "cat_match": (df["query_cat"] == df["item_cat_id"]).astype(np.int8),
        "mcat_match": (df["query_mcat"] == df["item_mcat_id"]).astype(np.int8),
        "loc_match": (df["query_loc"] == df["item_loc"]).astype(np.int8),

        "log_price": np.log1p(price),
        "click_conv": click,

        "len_query": qlen,
        "len_title": tlen,
        "len_desc": dlen,
        "len_q_t_ratio": qlen / (tlen + 1),
    })

    return feats, price_median

train_basic, price_median = build_basic(train)
test_basic, _ = build_basic(test, price_median)


# Здесь я получаю данные о том, насколько сильно у меня пересекаются значения в запросе с описанием и названием

In [ ]:
print("Fit TF-IDF...")
sample_n = min(500_000, len(train))
sample = train.sample(sample_n, random_state=RANDOM_SEED)

tfidf = TfidfVectorizer(max_features=40_000, min_df=10, ngram_range=(1,2), dtype=np.float32)
tfidf.fit(pd.concat([sample["query_text"], sample["item_title"], sample["item_description"]]))

def compute_tfidf_sims_batched(df, batch=200_000):
    sim_q_title = np.zeros(len(df), dtype=np.float32)
    sim_q_desc  = np.zeros(len(df), dtype=np.float32)
    sim_t_desc  = np.zeros(len(df), dtype=np.float32)

    for i in range(0, len(df), batch):
        sl = slice(i, i+batch)

        Q = tfidf.transform(df["query_text"].iloc[sl])
        T = tfidf.transform(df["item_title"].iloc[sl])
        D = tfidf.transform(df["item_description"].iloc[sl])

        sim_q_title[sl] = (Q.multiply(T)).sum(axis=1).A1
        sim_q_desc[sl]  = (Q.multiply(D)).sum(axis=1).A1

        del Q,T,D
        gc.collect()

    return pd.DataFrame({
        "sim_q_title": sim_q_title,
        "sim_q_desc": sim_q_desc,
    })

train_text = compute_tfidf_sims_batched(train)
test_text  = compute_tfidf_sims_batched(test)


# Вот тут я точно сделал OVERKILL по фичам, слишком много лишних и шумных вышло, но в моменте из-за количества фич получился неплохой score, т.е. модель действительно отработала лучше, чем без этих фич

In [ ]:
def add_group_stats(df_ids, feats, cols):
    joined = pd.concat([df_ids[["query_id"]].reset_index(drop=True),
                        feats.reset_index(drop=True)], axis=1)

    for c in cols:
        stats = joined.groupby("query_id")[c].agg(["mean","std","min","max"])
        joined = joined.join(stats, on="query_id")

        feats[f"{c}_z"] = (joined[c]-joined["mean"]) / (joined["std"]+1e-6)
        feats[f"{c}_minmax"] = (joined[c]-joined["min"]) / (joined["max"]-joined["min"]+1e-6)
        feats[f"{c}_rank"] = joined.groupby("query_id")[c].rank(ascending=False)
        feats[f"{c}_pct"] = joined.groupby("query_id")[c].rank(pct=True)

        joined.drop(columns=["mean","std","min","max"], inplace=True)

    return feats

train_feats = pd.concat([train_basic, train_text], axis=1)
test_feats  = pd.concat([test_basic, test_text], axis=1)

norm_cols = ["click_conv","log_price","sim_q_title","sim_q_desc"]
train_feats = add_group_stats(train[["query_id"]], train_feats, norm_cols)
test_feats  = add_group_stats(test[["query_id"]],  test_feats,  norm_cols)

# Тут я создал последнии категориальные фичи а также преобразовал их все в цифровые значения, которые будут совпадать в двух датасетах

In [ ]:
cat_cols = ["query_cat","query_mcat","query_loc","item_cat_id","item_mcat_id","item_loc"]

for c in cat_cols:
    full = pd.concat([train[c], test[c]])
    full = full.astype("category")
    train[c] = full.iloc[:len(train)]
    test[c]  = full.iloc[len(train):]

def cat_codes(df):
    return pd.DataFrame({c: df[c].cat.codes.astype(np.int32) for c in cat_cols})

train_cat = cat_codes(train)
test_cat  = cat_codes(test)

X = pd.concat([train_feats, train_cat], axis=1)
X_test = pd.concat([test_feats, test_cat], axis=1)

y = train["item_contact"].astype(np.float32).values
groups = train["query_id"].values

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0.0)

X_test = X_test.replace([np.inf, -np.inf], np.nan)
X_test = X_test.fillna(0.0)

assert np.isfinite(X.values).all()

In [ ]:
def sort_by_group(X, y, g):
    order = np.argsort(g, kind="mergesort")
    return X.iloc[order], y[order], g[order]

def group_sizes(g):
    return np.unique(g, return_counts=True)[1]


# Выставляю параметры для обучения, я впервые стокнулся с задачей ранжирования, но эти параметры показали себя наиболее оптимальными при обучении

In [ ]:
def lgb_params(use_gpu=True):
    params = dict(
        objective="lambdarank",
        metric="ndcg",
        ndcg_eval_at=[10],
        learning_rate=0.05,
        n_estimators=4000,
        num_leaves=511,
        min_data_in_leaf=200,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=5,
        reg_alpha=1,
        random_state=RANDOM_SEED,
        n_jobs=-1
    )
    if use_gpu:
        params.update({"device_type":"gpu","gpu_use_dp":False})
    return params

# Обучаю на train data

# делаем кросс валидацию с GroupKFold и LightGBM Ranker

In [ ]:
gkf = GroupKFold(n_splits=N_FOLDS)
models = []
best_iters = []

for fold,(tr,va) in enumerate(gkf.split(X,y,groups)):
    print(f"\nFold {fold+1}")

    X_tr,y_tr,g_tr = sort_by_group(X.iloc[tr], y[tr], groups[tr])
    X_va,y_va,g_va = sort_by_group(X.iloc[va], y[va], groups[va])

    model = lgb.LGBMRanker(**lgb_params(True))
    model.fit(
        X_tr,y_tr,
        group=group_sizes(g_tr),
        eval_set=[(X_va,y_va)],
        eval_group=[group_sizes(g_va)],
        callbacks=[lgb.early_stopping(200), lgb.log_evaluation(100)]
    )

    models.append(model)
    best_iters.append(model.best_iteration_)

# Делаем финальное обучение модели на полном датасете

In [ ]:
final_n = int(np.mean(best_iters))
final_model = lgb.LGBMRanker(**{**lgb_params(True),"n_estimators":final_n})

X_full,y_full,g_full = sort_by_group(X,y,groups)
final_model.fit(X_full,y_full,group=group_sizes(g_full))
models.append(final_model)

# Предсказываем на тестовой выборке усредненные значения по всем обученным моделям (ансамбль)
# На основе полученных скорингов формируется итоговый ranking внутри каждого query_id

In [ ]:
pred = np.zeros(len(X_test), dtype=np.float32)
for m in models:
    pred += m.predict(X_test)
pred /= len(models)

test["score"] = pred
submission = test.sort_values(["query_id","score"], ascending=[True,False])[["query_id","item_id"]]
submission.to_csv("solution_refactored.csv", index=False)

print("Saved solution_refactored.csv")